# KV Cache: don't recompute the past

> A faithful, runnable port of the concept page [`05-KV-Cache.md`](05-KV-Cache.md). The prose
> below is condensed from that page; the code cells are the same verified demo, executed here so
> the real speedup table is embedded as output.

Imagine writing an essay where, before you add each new word, you re-read the entire essay from the
very first word just to decide what comes next. Write word 500 and you've re-read 499 words; write
word 1,000 and you've re-read 999. That is *exactly* what a transformer does when it generates text
**without a KV cache**: every new token reruns attention over every token before it, recomputing the
same numbers it already computed a step ago.

**The cache is a speed-and-memory mechanism, not a modeling one.** With or without it, the model
produces the *identical* tokens — we prove that below — it only changes how fast, and how much VRAM,
that takes.

## The problem: autoregressive decoding repeats itself

LLMs generate **autoregressively** — one token at a time, each new token conditioned on every token
before it. To produce token $t$, the model forms a **query** for the current position and compares it
against a **key** for every previous position, then mixes the corresponding **values**.

The catch is the *next* step. Naively, to produce token $t+1$ you run the whole sequence through the
model again — recomputing the keys and values for tokens $1 \dots t$, *the very same K and V you
computed one step ago*.

> In a causal (decoder-only) transformer, a token's key and value depend **only on that token and the
> ones before it** — never on anything after. So once token 5's K and V are computed, they are
> **frozen for the rest of the generation**. Recomputing them is pure waste.

How much waste? At decode step $t$ the naive approach re-projects K and V for all $t$ positions; summed
over a sequence of length $n$ that's $1 + 2 + \dots + n = \tfrac{n(n+1)}{2}$ projections —
**quadratic** in sequence length. The cache turns each step into a single new projection — **linear**.

## What it is, and why K and V but never Q

A **KV cache** is a per-layer buffer storing the **key** and **value** vectors of every token processed
so far. Generation splits into two phases:

- **Prefill** — process the entire prompt in one parallel pass, computing and storing K and V for all
  prompt tokens. This fills the cache.
- **Decode** — generate one token at a time. Each step computes K and V for **only the single new
  token**, appends them to the cache, and attends over the whole cache.

**Why K and V, not Q?** When the model generates a new token, that token's *query* asks a question of
*all the past tokens' keys* and gathers *all the past tokens' values*. But the past tokens' queries have
already done their job and will never be asked again. So you keep what future tokens need (K and V) and
discard what's spent (past Q). The clean one-liner: **Q is per-step and disposable; K and V are
per-token and reused.**

## How it works: the cache tensor and the append

The cache for one layer is a pair of tensors shaped `[batch, n_kv_heads, seq_len, head_dim]` — one for
K, one for V — held **per layer**. Lifecycle:

1. **Prefill.** Run the prompt (length $N$) through every layer once; each layer projects K and V for
   all $N$ tokens and writes them into its cache.
2. **Decode step.** Take the single newest token; project its $q, k, v$; **append** $k, v$ to the
   cache (now length $N{+}1$); compute attention of the one query against the full cached K/V.
3. **Repeat** until end-of-sequence.

In a real engine "append" is an **in-place write into a pre-allocated buffer**, not a fresh allocation.
Growing the tensor with a `torch.cat` *every step* — as the teaching code below does for clarity — would
re-allocate and copy the whole cache each token, turning the $O(n)$ win back into an $O(n^2)$ disaster.
The note in `decode_with_cache` flags exactly this.

In [1]:
"""From-scratch KV cache: prove identical outputs, then time how the speedup GROWS with length.
Verified on Python 3.12 / torch 2.x, CPU."""
import time
import torch
import torch.nn.functional as F

torch.manual_seed(0)
d_model, n_heads, head_dim = 512, 8, 64
assert n_heads * head_dim == d_model
Wq = torch.randn(d_model, d_model) * 0.02
Wk = torch.randn(d_model, d_model) * 0.02
Wv = torch.randn(d_model, d_model) * 0.02


def split_heads(x):                 # (T, d_model) -> (n_heads, T, head_dim)
    T = x.shape[0]
    return x.view(T, n_heads, head_dim).transpose(0, 1)


def attn_step(q_t, K, V):           # q_t:(n_heads,1,head_dim)  K,V:(n_heads,T,head_dim)
    scores = (q_t @ K.transpose(-1, -2)) / head_dim ** 0.5   # query attends all cached keys
    return (F.softmax(scores, dim=-1) @ V).transpose(0, 1).reshape(1, d_model)


def decode_no_cache(emb, N):        # every step re-projects K,V for ALL tokens so far -> O(n^2)
    outs = []
    for t in range(1, N + 1):
        ctx = emb[:t]
        K, V = split_heads(ctx @ Wk), split_heads(ctx @ Wv)   # recomputed from scratch
        outs.append(attn_step(split_heads(emb[t - 1:t] @ Wq), K, V))
    return torch.cat(outs, 0)


def decode_with_cache(emb, N):      # project only the NEW token, append to the cache -> O(n)
    outs, K_cache, V_cache = [], None, None
    for t in range(1, N + 1):
        new = emb[t - 1:t]
        k_new, v_new = split_heads(new @ Wk), split_heads(new @ Wv)
        # NOTE: cat-per-step is the O(n^2) re-allocation trap; real engines write in place
        # into a pre-allocated buffer. Kept as cat here for clarity only.
        K_cache = k_new if K_cache is None else torch.cat([K_cache, k_new], dim=1)
        V_cache = v_new if V_cache is None else torch.cat([V_cache, v_new], dim=1)
        outs.append(attn_step(split_heads(new @ Wq), K_cache, V_cache))
    return torch.cat(outs, 0)


def timeit(fn, reps=3):
    fn()  # warmup
    t0 = time.perf_counter()
    for _ in range(reps):
        fn()
    return (time.perf_counter() - t0) / reps * 1e3


## Prove it, then watch the speedup grow

The loop below runs both paths at growing sequence lengths. Two things to read off the table:

1. the **`identical` column is `True` at every length** — the cache changes nothing about *what* the
   model produces; and
2. the **speedup widens** as the sequence grows — that widening *is* the $O(n^2) \to O(n)$ curve made
   visible. The no-cache loop's per-step work grows with position, so its total climbs faster than the
   cache's, and the ratio keeps opening up.

Trust the *shape* — a ratio that grows with length — not any single row's exact milliseconds.

In [2]:
print(f"{'N':>6} | {'no-cache':>10} | {'kv-cache':>10} | {'speedup':>8} | identical")
print("-" * 58)
for N in (256, 512, 1024, 2048):
    emb = torch.randn(N, d_model) * 0.1
    a, b = decode_no_cache(emb, N), decode_with_cache(emb, N)
    same = torch.allclose(a, b, atol=1e-5)
    ms_no = timeit(lambda: decode_no_cache(emb, N))
    ms_yes = timeit(lambda: decode_with_cache(emb, N))
    print(f"{N:>6} | {ms_no:>8.1f}ms | {ms_yes:>8.1f}ms | {ms_no / ms_yes:>6.1f}x | {same}")


     N |   no-cache |   kv-cache |  speedup | identical
----------------------------------------------------------


   256 |     61.2ms |     51.6ms |    1.2x | True


   512 |    188.0ms |    120.9ms |    1.6x | True


  1024 |    516.7ms |    311.1ms |    1.7x | True


  2048 |   1706.9ms |    860.4ms |    2.0x | True


## Try it yourself

Before you change anything, **predict**: the demo uses `n_heads=8, head_dim=64`. If you bump
`n_heads` to 16 (keeping `head_dim=64`, so `d_model` doubles to 1024), does the **speedup curve** —
the *ratio* of no-cache to kv-cache time at each `N` — shift up, shift down, or stay roughly the same?

Then run it and check. *Hint:* both paths do more work per step, but the *redundant* work the cache
removes still grows like $O(n^2)$ in `N` either way — so the curve's **shape** is driven by length,
not head count. Heavier heads change the absolute milliseconds, not the reason the ratio widens.

> To see the real thing, call `model.generate(..., use_cache=True)` vs `use_cache=False` in Hugging
> Face on a long generation and watch the wall-clock diverge. Same idea, full model.

**References and the full discussion (memory math, prefill vs decode, GQA/paging/quantization/windowing):
see [`05-KV-Cache.md`](05-KV-Cache.md) and [`05-KV-Cache.references.md`](05-KV-Cache.references.md).**